# Baseline probe : how good are the model's chess moves, measured in centipawns?


In [17]:
# Setup -- installs and clones on Colab, a no-op on a working local checkout.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/AmirBraham/chessllm.git"
REPO_DIR = "/content/chessllm"
N_POSITIONS = 50  # only used when data/positions.json has to be built

try:  # find_spec raises rather than returning None when `google` is absent
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False


def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    sh("apt-get -qq update && apt-get -qq install -y stockfish")
    sh(f"{sys.executable} -m pip install -q "
       f"'chess>=1.11.2' 'transformers>=5.14.1' pyarrow huggingface_hub")

    if not os.path.isdir(REPO_DIR):
        sh(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Debian puts the binary in /usr/games, which is not always on PATH here.
    if not shutil.which("stockfish") and os.path.exists("/usr/games/stockfish"):
        os.environ["PATH"] += os.pathsep + "/usr/games"

assert shutil.which("stockfish"), (
    "stockfish not on PATH -- `brew install stockfish` (mac) "
    "or `apt-get install -y stockfish` (linux)"
)

# The dataset is not in the repo; rebuilding it costs a few minutes of
# Stockfish time at depth 12.
if not os.path.exists("data/positions.json"):
    print(f"building data/positions.json ({N_POSITIONS} positions)...")
    sh(f"{sys.executable} build_data.py --n {N_POSITIONS}")

print(f"colab={IN_COLAB}  cwd={os.getcwd()}  stockfish={shutil.which('stockfish')}")

colab=True  cwd=/content/chessllm  stockfish=/usr/games/stockfish


In [ ]:
import json
import statistics as st

import chess

from board import build_prompt, parse_move
from engine import Engine
from qwen3 import generate, get_device, load

N = 20
MAX_NEW_TOKENS = 2048
THINK = True  # set False to disable Qwen3 thinking mode

records = json.load(open("data/positions.json"))[:N]
boards = [chess.Board(r["fen"]) for r in records]
print(f"{len(boards)} positions, device = {get_device()}")

100 positions, device = cuda


## What the model actually sees

In [19]:
print(build_prompt(boards[0]))

You are a chess engine. Choose the best move.

FEN: 2b2rk1/p1q4p/1n4p1/N1Npb3/1P2prP1/P6P/4QPB1/3R1RK1 w - - 0 29

  a b c d e f g h
8 . . b . . r k . 8
7 p . q . . . . p 7
6 . n . . . . p . 6
5 N . N p b . . . 5
4 . P . . p r P . 4
3 P . . . . . . P 3
2 . . . . Q P B . 2
1 . . . R . R K . 1
  a b c d e f g h

White to move.
Legal moves: Bf3 Bh1 Bxe4 Kh1 Kh2 Na4 Na6 Nab3 Nab7 Nc4 Nc6 Ncb3 Ncb7 Nd3 Nd7 Ne6 Nxe4 Qa2 Qa6 Qb2 Qb5 Qc2 Qc4 Qd2 Qd3 Qe1 Qe3 Qf3 Qxe4 Ra1 Rb1 Rc1 Rd2 Rd3 Rd4 Rde1 Rfe1 Rxd5 a4 b5 f3 g5 h4

Think briefly, then write the move on the last line in standard algebraic notation (SAN).


## Load the model

First run downloads ~1.2 GB from Hugging Face.

In [21]:
model, tok = load()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params on {model.device}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

596M params on cuda:0


## Generate

Watch the truncation count. Anything above zero means completions are hitting
the budget before committing to a move, and every move parsed out of those was
scraped from mid-reasoning rather than chosen.

In [22]:
prompts = [build_prompt(b) for b in boards]
texts, lengths = generate(
    model, tok, prompts, max_new_tokens=MAX_NEW_TOKENS, think=THINK
)

truncated = sum(n >= MAX_NEW_TOKENS for n in lengths)
print(f"mean {st.mean(lengths):.0f} tokens (max {max(lengths)})")
print(f"truncated: {truncated}/{len(lengths)}")
print(f"closed </think>: {sum('</think>' in t for t in texts)}/{len(texts)}")

mean 38 tokens (max 58)
truncated: 0/100
closed </think>: 0/100


## One completion in full

The interesting part is whether its reasoning describes the actual position.

In [23]:
print(texts[0])
print("\n" + "=" * 60)

move, raw = parse_move(boards[0], texts[0])
print(f"raw token : {raw!r}")
print(f"parsed    : {boards[0].san(move) if move else None}")
print(f"legal     : {move is not None}")

The best move for White is **Nc4**. This move creates a solid pawn line and threatens to cut off Black's king.

raw token : 'Nc4'
parsed    : Nc4
legal     : True


## Score all

`raw` is the token `parse_move` found; `move` is it validated against the
position. `raw` set with `move` empty means the model named an illegal move --
a different failure from naming none at all.

In [24]:
with Engine(depth=12, threads=1) as eng:
    rows = []
    for b, text, n_tok in zip(boards, texts, lengths):
        move, raw = parse_move(b, text)
        rows.append(
            {
                "raw": raw,
                "move": b.san(move) if move else None,
                "cp_loss": eng.cp_loss(b, move) if move else None,
                "best": b.san(eng.best(b)[0]),
                "tokens": n_tok,
                "trunc": n_tok >= MAX_NEW_TOKENS,
            }
        )

print(f"{'#':>2} {'raw':>12} {'move':>7} {'cp_loss':>8} {'best':>7} {'tok':>5}  trunc")
for i, r in enumerate(rows):
    cp = "-" if r["cp_loss"] is None else str(r["cp_loss"])
    print(
        f"{i:>2} {str(r['raw'])[:12]:>12} {str(r['move']):>7} {cp:>8} "
        f"{r['best']:>7} {r['tokens']:>5}  {r['trunc']}"
    )

 #          raw    move  cp_loss    best   tok  trunc
 0          Nc4     Nc4      627      f3    29  False
 1          Qd5     Qd5      406     Nh4    55  False
 2           b6      b6     1000     Bc3    43  False
 3          Rd3     Rd3       89     Rd6    44  False
 4          Qc8     Qc8      290     Nc8    41  False
 5          Rc1    Rc1+        1    Rc1+    45  False
 6          Nc5     Nc5        0     Nc5    36  False
 7          Nc3     Nc3      196     Ng5    41  False
 8           c5      c5      377     Be7    36  False
 9           f5      f5        0     Kf3    24  False
10          Qf5     Qf5     1000      f4    35  False
11         Bxf6    Bxf6        0    Bxf6    33  False
12          Qh4     Qh4      887    Qxf1    53  False
13          Qc5    Qc5+        0    Qc5+    52  False
14          Rc3     Rc3     1000    gxh4    41  False
15          Qb3     Qb3      127    Qxb7    40  False
16          Bd2     Bd2      173      f3    41  False
17           b3      b3     

## Against the floor and the ceiling

A cp_loss number means nothing on its own. Random is the floor to beat;
Stockfish's own move is the ceiling, and lands near 10-20 rather than 0 because
the search runs one ply deeper after the move than at the root.

In [25]:
from baseline import run_random, run_stockfish, summarize

with Engine(depth=12, threads=1) as eng:
    summarize("random legal move", run_random(eng, boards))
    summarize("stockfish best", run_stockfish(eng, boards))

losses = [r["cp_loss"] for r in rows if r["cp_loss"] is not None]
print(f"\nqwen3-0.6B  ({len(losses)} legal of {len(rows)})")
if losses:
    print(f"  mean cp_loss     {st.mean(losses):6.0f}")
    print(f"  median cp_loss   {st.median(losses):6.0f}")
    print(f"  good moves <=50  {sum(x <= 50 for x in losses) / len(losses):6.1%}")


random legal move  (n=100)
  mean cp_loss        598
  median cp_loss      636
  good moves <=50cp    7.0%

stockfish best  (n=100)
  mean cp_loss         24
  median cp_loss        2
  good moves <=50cp   80.0%

qwen3-0.6B  (94 legal of 100)
  mean cp_loss        524
  median cp_loss      436
  good moves <=50   12.8%


## Where to go from here

- **High truncation** -- raise `MAX_NEW_TOKENS`, or set `THINK = False` and rerun.
  No-think should collapse completions to a few dozen tokens.
- **Low legal rate** -- the legal-move list is in the prompt, so this means the
  model is not reading it. Try `build_prompt(b, include_board=False)` to see
  whether the ASCII board is distracting rather than helping.
- **cp_loss near the random floor** -- expected for the untrained baseline. That
  gap is the room GRPO has to work with.